# Ứng dụng Generative AI trong công việc Data Analyst

**Mục tiêu:** kết hợp Python/pandas với GenAI theo quy trình:

> Data → Python Analysis → GenAI Assistance → Human Verification → Business Decision

Notebook này **không gọi API GenAI**. Sinh viên copy các prompt được cung cấp sang ChatGPT/Gemini/Claude,
sau đó đưa kết quả đã kiểm chứng trở lại notebook/báo cáo.


## Quy tắc sử dụng GenAI trong bài

1. Không tải dữ liệu nhạy cảm hoặc thông tin cá nhân lên công cụ GenAI công cộng.
2. AI có thể sinh code nhưng sinh viên phải chạy và kiểm tra.
3. AI có thể diễn giải nhưng chỉ số liệu do Python tính mới được xem là nguồn kiểm chứng.
4. Không chấp nhận kết luận nếu dữ liệu không hỗ trợ.
5. Lưu Prompt Log.


## 1. Import thư viện


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.options.display.float_format = "{:,.2f}".format


## 2. Đọc dữ liệu


In [ ]:
df = pd.read_csv("TH02_business_transactions_genai.csv")
df.head()


### Kiểm tra kích thước, kiểu dữ liệu và missing


In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


## Hoạt động GenAI P01: Lập kế hoạch EDA

Mở file **Bo_prompt_mau_GenAI_Data_Analyst.md**, dùng prompt **P01**.

### Nhiệm vụ
- So sánh kế hoạch EDA của AI với các phép kiểm tra ở trên.
- Ghi vào Prompt Log ít nhất 1 điểm AI đề xuất hợp lý và 1 điểm cần kiểm chứng.


## 3. Kiểm tra dữ liệu bất thường


In [ ]:
checks = {
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "UnitPrice <= 0": (df["UnitPrice"] <= 0).sum(),
    "UnitCost <= 0": (pd.to_numeric(df["UnitCost"], errors="coerce") <= 0).sum(),
}
checks


### Thảo luận
Không phải mọi outlier đều là lỗi.  
Ví dụ một đơn hàng có Quantity rất lớn có thể là khách hàng doanh nghiệp.

Hãy dùng **P03** để hỏi AI về checklist làm sạch, sau đó đánh dấu:
- ✓ hợp lý
- ? cần kiểm chứng
- ✗ không phù hợp


## 4. Làm sạch cơ bản


In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Bỏ duplicate hoàn toàn
df = df.drop_duplicates().copy()

# Điền missing cho biến phân loại
df["Region"] = df["Region"].fillna("Unknown")
df["Channel"] = df["Channel"].fillna("Unknown")

# UnitCost: điền median theo Product
df["UnitCost"] = (
    df.groupby("Product")["UnitCost"]
      .transform(lambda s: s.fillna(s.median()))
)

df.info()


> Lưu ý: xử lý `Quantity <= 0` và `UnitPrice <= 0` cần quyết định nghiệp vụ.
> Trong notebook này ta tạo một DataFrame sạch dùng cho KPI bằng cách chỉ giữ giao dịch có số lượng và giá bán dương.


In [ ]:
df_clean = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["UnitCost"] > 0)
].copy()

print("Rows after business-rule filtering:", len(df_clean))


## 5. Tạo KPI: kiểm chứng code do AI sinh


In [ ]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["Cost"] = df_clean["Quantity"] * df_clean["UnitCost"]
df_clean["Profit"] = df_clean["Revenue"] - df_clean["Cost"]
df_clean["ProfitMargin"] = np.where(
    df_clean["Revenue"] != 0,
    df_clean["Profit"] / df_clean["Revenue"],
    np.nan
)

df_clean[["Revenue","Cost","Profit","ProfitMargin"]].describe()


### Hoạt động GenAI P02
Dùng prompt **P02** và so sánh code AI sinh với code ở trên.

Trả lời:
1. Code có xử lý chia cho 0 không?
2. Có làm mất missing value không?
3. Có thêm giả định nghiệp vụ nào không?


## 6. KPI tổng quan


In [ ]:
kpis = {
    "Total Revenue": df_clean["Revenue"].sum(),
    "Total Profit": df_clean["Profit"].sum(),
    "Transactions": df_clean["TransactionID"].nunique(),
    "Customers": df_clean["CustomerID"].nunique(),
    "Overall Profit Margin": df_clean["Profit"].sum() / df_clean["Revenue"].sum()
}
kpis


## 7. Phân tích theo Region


In [ ]:
region_summary = (
    df_clean.groupby("Region")
    .agg(
        Revenue=("Revenue","sum"),
        Profit=("Profit","sum"),
        Transactions=("TransactionID","nunique"),
        Customers=("CustomerID","nunique")
    )
)
region_summary["ProfitMargin"] = region_summary["Profit"] / region_summary["Revenue"]
region_summary.sort_values("Revenue", ascending=False)


### Hoạt động GenAI P05: Diễn giải insight
Copy bảng `region_summary` sang AI bằng prompt **P05**.

Đối với mỗi insight AI tạo ra, gắn nhãn:
- `[SUPPORTED]`
- `[NEEDS CHECK]`
- `[UNSUPPORTED]`


## 8. Phân tích theo Channel


In [ ]:
channel_summary = (
    df_clean.groupby("Channel")
    .agg(
        Revenue=("Revenue","sum"),
        Profit=("Profit","sum"),
        Transactions=("TransactionID","nunique")
    )
)
channel_summary["ProfitMargin"] = channel_summary["Profit"] / channel_summary["Revenue"]
channel_summary


## 9. Trực quan hóa – kiểm chứng đề xuất của AI


In [ ]:
plot_df = region_summary.sort_values("Revenue", ascending=False)

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(plot_df.index, plot_df["Revenue"])
ax.set_title("Revenue by Region")
ax.set_xlabel("Region")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(channel_summary.index, channel_summary["ProfitMargin"])
ax.set_title("Profit Margin by Channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Profit Margin")
plt.tight_layout()
plt.show()


### Hoạt động GenAI P06
Dùng prompt **P06** để AI đề xuất biểu đồ.
Sau đó:
1. chạy code AI sinh;
2. so sánh với biểu đồ trong notebook;
3. nhận xét chart nào dễ hiểu hơn và vì sao.


## 10. Phân tích khách hàng: Pareto


In [ ]:
customer = (
    df_clean.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

customer["CumRevenue"] = customer["Revenue"].cumsum()
customer["CumPct"] = customer["CumRevenue"] / customer["Revenue"].sum()

share_customers_80 = (customer["CumPct"] <= 0.80).mean()
print(f"Tỷ lệ khách hàng tạo ra gần 80% doanh thu: {share_customers_80:.2%}")
customer.head(10)


## 11. Dữ liệu phi cấu trúc: ReviewText

GenAI đặc biệt hữu ích khi dữ liệu là văn bản.
Ta lấy một mẫu review để sinh viên dùng prompt **P07**.


In [ ]:
sample_reviews = (
    df_clean["ReviewText"]
    .dropna()
    .sample(30, random_state=42)
    .tolist()
)

for i, review in enumerate(sample_reviews, 1):
    print(f"{i:02d}. {review}")


### Hoạt động GenAI P07
Copy 20–30 review trên vào AI.

Yêu cầu:
- nhóm tối đa 5 chủ đề;
- sentiment;
- ví dụ đại diện;
- **không được suy tỷ lệ toàn dataset từ mẫu**.

Sau đó kiểm tra xem AI có vi phạm điều kiện cuối không.


## 12. Kiểm tra sentiment bằng dữ liệu có cấu trúc


In [ ]:
score_summary = (
    df_clean.groupby("SatisfactionScore")
    .size()
    .rename("Count")
    .to_frame()
)
score_summary["Pct"] = score_summary["Count"] / score_summary["Count"].sum()
score_summary


So sánh kết quả AI phân tích ReviewText với `SatisfactionScore`.
Hai nguồn có nhất quán không? Nếu không, cần giải thích là đây chỉ là mẫu review chứ không phải toàn bộ dữ liệu.


## 13. Debug với GenAI


In [ ]:
# Ví dụ đúng
df_clean.groupby("Region")["Revenue"].mean()


### Bài debug
Cố ý sửa dòng trên thành:

```python
df_clean.groupby("Region")["Revenue"].mean
```

Quan sát kết quả và dùng **P04** để yêu cầu AI giải thích vì sao thiếu `()` là vấn đề.


## 14. Hallucination Test


Giả sử chỉ cung cấp cho AI bảng gồm `Region`, `Revenue`, `Profit`.

Dùng prompt **P11**:

> “Khu vực nào có tỷ lệ khách hàng quay lại cao nhất?”

Câu trả lời đúng phải là **không đủ dữ liệu** vì thiếu CustomerID/lịch sử mua lặp lại.

Ghi lại phản hồi của AI vào Prompt Log.


## 15. Executive Summary


In [ ]:
verified_summary = {
    "TotalRevenue": df_clean["Revenue"].sum(),
    "TotalProfit": df_clean["Profit"].sum(),
    "BestRegionByRevenue": region_summary["Revenue"].idxmax(),
    "BestChannelByMargin": channel_summary["ProfitMargin"].idxmax(),
}
verified_summary


Dùng **P08** và chỉ đưa các KPI đã kiểm chứng ở trên.

Sau khi AI viết Executive Summary:
- tô/đánh dấu mọi con số;
- đối chiếu lại với notebook;
- xóa hoặc sửa mọi nguyên nhân mà dữ liệu chưa chứng minh.


## 16. Công việc văn phòng với GenAI


Chọn **một** trong các prompt:
- P09: Email báo cáo
- P10: Outline 5 slide

Yêu cầu: nội dung phải dựa trên kết quả đã kiểm chứng trong notebook.


## 17. Prompt Self-check


Dùng **P13** để AI tự kiểm tra chính câu trả lời của nó.

So sánh:
- AI tự phát hiện được bao nhiêu vấn đề?
- Có vấn đề nào sinh viên phát hiện nhưng AI không phát hiện?


## 18. Prompt Log: sinh viên điền

| ID | Prompt | AI trả lời tóm tắt | Vấn đề phát hiện | Prompt cải tiến | Cách kiểm chứng |
|---|---|---|---|---|---|
| P01 |  |  |  |  |  |
| P02 |  |  |  |  |  |
| P05 |  |  |  |  |  |
| P11 |  |  |  |  |  |


## 19. Kết luận cuối bài


Viết 150–250 từ trả lời:

1. GenAI giúp Data Analyst tiết kiệm thời gian ở bước nào?
2. Bước nào vẫn bắt buộc con người kiểm chứng?
3. Ví dụ hallucination hoặc suy diễn mà nhóm phát hiện.
4. Một nguyên tắc sử dụng GenAI mà nhóm cho là quan trọng nhất.


## 20. Checklist nộp bài


- [ ] Notebook chạy từ đầu đến cuối
- [ ] Có ít nhất 4 prompt trong Prompt Log
- [ ] Có 1 ví dụ debug bằng GenAI
- [ ] Có 1 hallucination test
- [ ] Có ít nhất 2 biểu đồ
- [ ] Có phân tích ReviewText
- [ ] Có Executive Summary
- [ ] Có kết luận về giới hạn của GenAI
